# Import libraries and initialize the llm

In [2]:
import os
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [3]:
api_key = os.getenv("API_KEY")
llm = model = GoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=1500,
    timeout=None,
    max_retries=2
)

# Add memory to the indication

Based on adding the conversation history to each prompt. You can add either the whole conversation or a summary of the conversation (summarized with an LLM). You can also either use the context window or combine the previous strategies, adding both a summary and the last messages.

## Save the whole conversation history

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Your name is {name}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

conversational_chain

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x0000025B894D76A0>, input_messages_key='input', history_messages_key='history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [10]:
# Invoke the chain, passing a session ID to track the user
response1 = conversational_chain.invoke(
    {
        "name":"AI Bot",
        "input": "Hi, my name is Alex."
    },
    config={"configurable": {"session_id": "session_123"}}
)
print(f"Response 1: {response1}")

# Using the same session_id
response2 = conversational_chain.invoke(
    {
        "name":"AI Bot",
        "input": "What is my name? What is your name?"
    },
    config={"configurable": {"session_id": "session_123"}}
)
print(f"Response 2: {response2}")

# Using a different session_id
response3 = conversational_chain.invoke(
    {
        "name":"Arturito",
        "input": "What is my name? What is your name?"
    },
    config={"configurable": {"session_id": "session_124"}}
)
print(f"Response 3: {response3}")

Response 1: AI: Hi Alex, it's nice to meet you! I'm AI Bot. How can I help you today?
Response 2: AI: Your name is Alex, and my name is AI Bot.
Response 3: I do not have access to your personal information, so I cannot tell you your name. My name is Arturito.


In [11]:
store

{'session_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi, my name is Alex.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hi Alex, it's nice to meet you! I'm AI Bot. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Hi, my name is Alex.', additional_kwargs={}, response_metadata={}), AIMessage(content="AI: Hi Alex, it's nice to meet you! I'm AI Bot. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name? What is your name?', additional_kwargs={}, response_metadata={}), AIMessage(content='AI: Your name is Alex, and my name is AI Bot.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'session_124': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is my name? What is your name?', additional_kwargs={}, response_metadata={}), 